# Demo 1 — Event Hub Producer

This notebook reads current crypto prices from the Binance public API and sends JSON events to Azure Event Hubs.

It demonstrates:
- calling a public REST API
- creating JSON events
- reading the Event Hub connection string from Databricks secrets
- sending events to Event Hubs
- adding metadata such as `event_id`, `event_time`, `producer_id`, and `source_system`

The first event schema is intentionally simple. Extra fields will be added later during the schema-evolution demo.


## 1. Load shared configuration

The Event Hub name, secret settings, symbols, and source-system name come from `config/00_config`.


In [0]:
%run ../config/00_config


## 2. Install the Event Hub client library

Run this once when `azure-eventhub` is not already installed. Databricks may ask you to restart Python; if so, rerun the notebook from the beginning.


In [0]:
%pip install azure-eventhub


## 3. Import required libraries


In [0]:
import json
import time
import uuid
from datetime import datetime, timezone
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

from azure.eventhub import EventData, EventHubProducerClient


## 4. Producer settings

The defaults send one event per symbol for 10 cycles:

- 3 symbols
- 10 cycles
- 30 total events
- 5 seconds between cycles


In [0]:
binance_price_url = "https://data-api.binance.vision/api/v3/ticker/price"

producer_id = "demo1_binance_price_producer"
number_of_cycles = 10
polling_interval_seconds = 5
request_timeout_seconds = 15

print(f"Symbols: {historical_symbols}")
print(f"Event Hub: {eventhub_name}")
print(f"Producer ID: {producer_id}")
print(f"Cycles: {number_of_cycles}")
print(f"Polling interval: {polling_interval_seconds} seconds")


## 5. Retrieve the Event Hub connection string

The secret value is retrieved but never printed.


In [0]:
try:
    eventhub_connection_string = dbutils.secrets.get(
        scope=eventhub_secret_scope,
        key=eventhub_secret_key,
    )
except Exception as exc:
    raise RuntimeError(
        "Could not retrieve the Event Hub connection string. "
        f"Check secret scope '{eventhub_secret_scope}' and "
        f"secret key '{eventhub_secret_key}'."
    ) from exc

if not eventhub_connection_string:
    raise RuntimeError("The Event Hub connection string is empty.")

print("Event Hub connection string retrieved successfully.")


## 6. Function to get the latest Binance price


In [0]:
def get_latest_price(symbol: str) -> float:
    request_url = f"{binance_price_url}?{urlencode({'symbol': symbol})}"

    try:
        with urlopen(
            request_url,
            timeout=request_timeout_seconds,
        ) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        raise RuntimeError(
            f"Binance returned HTTP {exc.code} for {symbol}."
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            f"Could not reach Binance for {symbol}: {exc.reason}"
        ) from exc

    if "price" not in payload:
        raise ValueError(
            f"Binance response does not contain price for {symbol}: {payload}"
        )

    return float(payload["price"])


## 7. Create the initial event structure

Each event contains:
- `event_id`
- `symbol`
- `price_usd`
- `event_time`
- `producer_id`
- `source_system`


In [0]:
def create_price_event(symbol: str, price_usd: float) -> dict:
    return {
        "event_id": str(uuid.uuid4()),
        "symbol": symbol,
        "price_usd": price_usd,
        "event_time": datetime.now(timezone.utc).isoformat(),
        "producer_id": producer_id,
        "source_system": source_system_streaming,
    }

sample_event = create_price_event("BTCUSDT", 0.0)
print(json.dumps(sample_event, indent=2))


## 8. Test the Binance API


In [0]:
api_test_results = []

for symbol in historical_symbols:
    latest_price = get_latest_price(symbol)
    api_test_results.append({
        "symbol": symbol,
        "price_usd": latest_price,
        "status": "READY",
    })

display(spark.createDataFrame(api_test_results))


## 9. Create the Event Hub producer client


In [0]:
try:
    producer_client = EventHubProducerClient.from_connection_string(
        conn_str=eventhub_connection_string,
        eventhub_name=eventhub_name,
    )
except Exception as exc:
    raise RuntimeError(
        f"Could not create producer client for Event Hub '{eventhub_name}'."
    ) from exc

print("Event Hub producer client created successfully.")


## 10. Send live price events

Each cycle sends BTC, ETH, and SOL events in one Event Hub batch.


In [0]:
sent_events = []

try:
    with producer_client:
        for cycle_number in range(1, number_of_cycles + 1):
            event_batch = producer_client.create_batch()
            cycle_events = []

            for symbol in historical_symbols:
                latest_price = get_latest_price(symbol)
                event = create_price_event(symbol, latest_price)

                event_batch.add(EventData(json.dumps(event)))
                cycle_events.append(event)

            producer_client.send_batch(event_batch)
            sent_events.extend(cycle_events)

            print(
                f"Cycle {cycle_number}/{number_of_cycles}: "
                f"sent {len(cycle_events)} events"
            )

            if cycle_number < number_of_cycles:
                time.sleep(polling_interval_seconds)

except Exception as exc:
    raise RuntimeError(
        "Event production failed. Check the Event Hub connection, "
        "permissions, network access, and API availability."
    ) from exc


## 11. Display sent events


In [0]:
if not sent_events:
    raise RuntimeError("No events were sent.")

sent_events_df = spark.createDataFrame(sent_events)

display(
    sent_events_df.orderBy("event_time", "symbol")
)

print("Event Hub producer completed successfully.")
print(f"Total events sent: {len(sent_events)}")
print(f"Expected events: {number_of_cycles * len(historical_symbols)}")


## Next notebook

`streaming/05_eventhub_consumer.ipynb`

The consumer will read these JSON events from Event Hubs and write them into the permanent streaming Bronze Delta table.
